In [57]:
from google.colab import drive
drive.mount("/content/drive")

import torch
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| cihaz:", device)
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name, f"| bellek: {props.total_memory / 2**30:.1f} GB")

ROOT = Path("/content/drive/MyDrive/deneme")
VIDEO_PATH = Path("/content/drive/MyDrive/0.mkv")
MODEL_DIR = ROOT / "models"
TARGET_CACHE = ROOT / "checkpoints" / "hnerv384_targets_evalrgb_v1.pt"
OUT_DIR = ROOT / "b2"                      # bu notebook'un bütün çıktıları buraya
OUT_DIR.mkdir(exist_ok=True)

for p in (VIDEO_PATH, MODEL_DIR / "posenet.safetensors", MODEL_DIR / "segnet.safetensors", TARGET_CACHE):
    print("OK " if p.exists() else "YOK", p)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch: 2.11.0+cu130 | cihaz: cuda
GPU: NVIDIA L4 | bellek: 22.0 GB
OK  /content/drive/MyDrive/0.mkv
OK  /content/drive/MyDrive/deneme/models/posenet.safetensors
OK  /content/drive/MyDrive/deneme/models/segnet.safetensors
OK  /content/drive/MyDrive/deneme/checkpoints/hnerv384_targets_evalrgb_v1.pt


In [56]:
gpu_temizle()

/tmp/ipykernel_4019/3881937137.py:30: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(o, types.TracebackType):


silindi: counts, distortion_net, g16, ids, loss, model, net_params, opt, p, perm
Flushing output cache (0 entries)


In [58]:
import gc, sys, traceback, types, torch, torch.nn as nn

KORU = {"segnet", "posenet", "seg_targets", "pose_targets", "PILOT", "device"}  # silinmeyecekler

def _gpu_tutuyor(o, d=0):
    if torch.is_tensor(o): return o.is_cuda
    if isinstance(o, nn.Module): return any(t.is_cuda for t in [*o.parameters(), *o.buffers()])
    if isinstance(o, torch.optim.Optimizer):
        return any(torch.is_tensor(v) and v.is_cuda for s in o.state.values() for v in s.values()) \
            or any(p.is_cuda for g in o.param_groups for p in g["params"])
    if d < 2 and isinstance(o, dict): return any(_gpu_tutuyor(v, d + 1) for v in list(o.values())[:500])
    if d < 2 and isinstance(o, (list, tuple)): return any(_gpu_tutuyor(v, d + 1) for v in list(o)[:500])
    return False

def _kareleri_bosalt(exc, gorulen=None):
    gorulen = gorulen or set()
    while exc is not None and id(exc) not in gorulen:            # hata zinciri: cause / context
        gorulen.add(id(exc))
        if exc.__traceback__ is not None:
            traceback.clear_frames(exc.__traceback__)            # karelerdeki yerel değişkenleri sil
        _kareleri_bosalt(exc.__cause__, gorulen)
        exc = exc.__context__

def gpu_temizle(*isimler, koru=()):
    """gpu_temizle() → GPU tutan tüm globalleri sil (KORU hariç). İsim verirsen onları da siler."""
    try: torch.cuda.synchronize()                                # yarıda kalan GPU işini bitir
    except Exception: pass
    # 1) donmuş traceback kareleri (Durdur / OOM / hata)
    for o in gc.get_objects():                                   # kim saklarsa saklasın: tüm traceback'ler
      if isinstance(o, types.TracebackType):
          traceback.clear_frames(o)                             # karelerin yerellerini boşalt
    ip = get_ipython()
    hatalar = [getattr(sys, "last_exc", None), getattr(sys, "last_value", None)]
    if getattr(ip, "last_execution_result", None) is not None:
        hatalar.append(ip.last_execution_result.error_in_exec)
        ip.last_execution_result.error_in_exec = None
    for e in hatalar:
        if isinstance(e, BaseException): _kareleri_bosalt(e)
    if getattr(sys, "last_traceback", None) is not None:
        traceback.clear_frames(sys.last_traceback)
    sys.last_exc = sys.last_type = sys.last_value = sys.last_traceback = None
    # 2) globaller: verilen isimler + GPU tutan her şey (korunanlar hariç)
    g = ip.user_ns
    sil = set(isimler) | {n for n, v in list(g.items())
                          if not n.startswith("_") and n not in KORU | set(koru) and _gpu_tutuyor(v)}
    for n in sorted(sil):
        g.pop(n, None)
    print("silindi:", ", ".join(sorted(sil)) or "—")
    # 3) Out[] geçmişi ve _ / __ / ___
    ip.run_line_magic("reset", "-f out")
    # 4) topla ve rafı geri ver
    gc.collect()
    torch.cuda.empty_cache(); torch.cuda.ipc_collect()

# **SEGNET POSENET YUKLEME **

In [59]:
!uv pip -q install segmentation-models-pytorch timm einops safetensors av

import sys
sys.path.insert(0, str(ROOT))                      # research/ klasörü Drive'da, deneme/ altında
from research.modules import DistortionNet
from research.oracle_pair import seg_preprocess_grad, pose_preprocess_grad

distortion_net = DistortionNet().eval().to(device)
distortion_net.load_state_dicts(MODEL_DIR / "posenet.safetensors", MODEL_DIR / "segnet.safetensors", device)
distortion_net.requires_grad_(False)               # ağırlıkları asla değişmeyecek

n_seg = sum(p.numel() for p in distortion_net.segnet.parameters())
n_pose = sum(p.numel() for p in distortion_net.posenet.parameters())
print(f"SegNet {n_seg / 1e6:.1f}M | PoseNet {n_pose / 1e6:.1f}M parametre (donmuş)")

SegNet 9.5M | PoseNet 13.9M parametre (donmuş)


In [60]:
targets = torch.load(TARGET_CACHE, map_location="cpu", weights_only=True)
seg_targets = targets["seg_class"].to(device, dtype=torch.uint8)    # [600, 384, 512]: piksel başına sınıf
pose_targets = targets["pose"].to(device, dtype=torch.float32)      # [600, 6]: pair başına kamera hareketi
NUM_PAIRS = seg_targets.shape[0]
print("Seg:", tuple(seg_targets.shape), "| Pose:", tuple(pose_targets.shape), "| pair:", NUM_PAIRS)

counts = torch.bincount(seg_targets.flatten().long(), minlength=5)
for c, n in enumerate(counts.tolist()):
    print(f"  class {c}: %{100 * n / counts.sum().item():6.2f}")
print("Pose std (boyut başına):", [round(v, 4) for v in pose_targets.std(0).tolist()])

Seg: (600, 384, 512) | Pose: (600, 6) | pair: 600
  class 0: % 23.23
  class 1: %  0.59
  class 2: % 49.52
  class 3: %  1.24
  class 4: % 25.43
Pose std (boyut başına): [1.2574, 0.0358, 0.0299, 0.0096, 0.0074, 0.0286]


In [61]:
import torch.nn.functional as F

def inflate(native):
    """[B, 2, 3, 384, 512] → [B, 2, 3, 874, 1164]. İleri geçişte yuvarlar, geri geçişte gradyanı geçirir."""
    b = native.shape[0]
    up = F.interpolate(native.flatten(0, 1), size=(874, 1164), mode="bicubic", align_corners=False)
    up = up.clamp(0, 255)
    up = up + (up.round() - up).detach()          # STE: ileri = round, geri = özdeşlik
    return up.view(b, 2, 3, 874, 1164)


@torch.no_grad()
def score(render, ids_all, bs=16):
    """render(ids) → [B, 2, 3, 384, 512] (0–255). Resmi ölçümle aynı ayar: FP32, TF32 kapalı."""
    old = (torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32)
    torch.backends.cuda.matmul.allow_tf32 = torch.backends.cudnn.allow_tf32 = False
    wrong, pix, pose_sq = 0, 0, 0.0
    for s in range(0, len(ids_all), bs):
        ids = ids_all[s:s + bs]
        full = inflate(render(ids))
        pred = distortion_net.segnet(seg_preprocess_grad(full)).argmax(1)
        pose = distortion_net.posenet(pose_preprocess_grad(full))["pose"][:, :6].float()
        wrong += int((pred != seg_targets[ids].long()).sum())
        pix += pred.numel()
        pose_sq += float((pose - pose_targets[ids]).square().mean(1).sum())
    torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32 = old
    seg, pose_t = 100 * wrong / pix, (10 * pose_sq / len(ids_all)) ** 0.5
    return {"dist": seg + pose_t, "seg": seg, "pose": pose_t}


PILOT = torch.arange(60, device=device)          # pilot: ilk 60 pair (ardışık, zamansal bağlam için)
gray = lambda ids: torch.full((len(ids), 2, 3, 384, 512), 128.0, device=device)
print("Düz gri kare (en kötü referans):", score(gray, PILOT))

Düz gri kare (en kötü referans): {'dist': 93.41212415131939, 'seg': 50.60173034667969, 'pose': 42.8103938046397}


# **MODEL**

In [62]:
import math
import torch.nn as nn
GRID_LEVELS = {64:2, 32:2, 16:2}
WIDTHS = [16, 16, 16, 16, 12, 12, 8]

class GridDecoder(nn.Module):
  def __init__(self, n_pairs, levels = GRID_LEVELS, widths = WIDTHS):
    super().__init__()
    self.grids = nn.ParameterDict({str(d): nn.Parameter(torch.zeros(n_pairs, c, 384 // d, 512 // d))
                                  for d, c in levels.items()})

    self.inject = nn.ModuleDict({str(d): nn.Conv2d(c, widths[int(math.log2(64 // d))], 1)
                                     for d, c in levels.items()})
    self.ups = nn.ModuleList(nn.Conv2d(widths[i], 4 * widths[i + 1], 3, padding=1) for i in range(6))
    self.skips = nn.ModuleList(nn.Conv2d(widths[i], widths[i + 1], 1) for i in range(6))
    self.head = nn.Conv2d(widths[-1], 6, 3, padding=1)
  def forward(self, ids, grids = None):
    g = grids if grids is not None else {k: v[ids] for k, v in self.grids.items()}
    x = self.inject["64"](g["64"])
    for i in range(6):
      up = F.pixel_shuffle(self.ups[i](x), 2)
      skip = F.interpolate(self.skips[i](x), scale_factor = 2, mode = "bilinear", align_corners = False)
      x = F.gelu(up + skip)
      d = str(64//2 ** (i+1))
      if d in self.inject:
        x = x + self.inject[d](g[d])
    return (torch.sigmoid(self.head(x)) * 255).view(len(ids), 2, 3, 384, 512)

In [63]:
model = GridDecoder(NUM_PAIRS).to(device)
n_net = sum(p.numel() for n, p in model.named_parameters() if not n.startswith("grids"))
n_grid = sum(g[0].numel() for g in model.grids.values())
print(f"Sentez ağı: {n_net:,} parametre (8 bit ≈ {n_net / 1e3:.1f} KB)")
print(f"Izgara: pair başına {n_grid:,} değer → ~1 bit/değer ise pair başına {n_grid / 8:.0f} bayt, "
      f"600 pair ≈ {600 * n_grid / 8 / 1e3:.0f} KB")
print("Çıktı:", tuple(model(PILOT[:4]).shape), "| eğitimsiz skor:", score(lambda ids: model(ids), PILOT))

Sentez ağı: 45,382 parametre (8 bit ≈ 45.4 KB)
Izgara: pair başına 2,016 değer → ~1 bit/değer ise pair başına 252 bayt, 600 pair ≈ 151 KB
Çıktı: (4, 2, 3, 384, 512) | eğitimsiz skor: {'dist': 92.79104502601501, 'seg': 50.60173034667969, 'pose': 42.189314679335325}


# **LOSS**

In [53]:
def l7_loss(logits, truth, tau = 0.3,thr =1.0, mult = 4.0):
  correct = logits.gather(1, truth.unsqueeze(1))
  others = logits.clone()
  others.scatter_(1, truth.unsqueeze(1), torch.finfo(others.dtype).min)
  margin = correct - others.amax(1, keepdim = True)
  per_pixel = tau * F.softplus(-margin / tau)

  with torch.no_grad():
    w = 1.0 + mult * (margin < thr).float()
    w = w/w.mean()

  return (per_pixel * w).mean()

def task_loss(native, ids):
  full = inflate(native)
  logits = distortion_net.segnet(seg_preprocess_grad(full))
  pose = distortion_net.posenet(pose_preprocess_grad(full))["pose"][:, :6].float()
  seg_l = 100 * l7_loss(logits, seg_targets[ids].long())
  pose_l = torch.sqrt(10 * F.mse_loss(pose, pose_targets[ids]) + 1e-6)
  return seg_l + pose_l, seg_l.item(), pose_l.item()

In [64]:
ids = PILOT[:8]
model.zero_grad(set_to_none=True)
loss, sl, pl = task_loss(model(ids), ids)
loss.backward()
print(f"loss = {loss.item():.3f}  (seg {sl:.3f} + pose {pl:.3f})")

groups = {}
for name, p in model.named_parameters():
    groups.setdefault(name.split(".")[0], []).append(0.0 if p.grad is None else p.grad.norm().item())
for g, norms in groups.items():
    print(f"  {g:<7} gradyanı sıfır olmayan: {sum(n > 0 for n in norms)}/{len(norms)} | en küçük norm {min(norms):.1e}")

g16 = model.grids["16"].grad
print("Izgara gradyanı — batch'teki pair'ler:", f"{g16[:8].abs().sum().item():.2e}",
      "| batch dışındakiler:", f"{g16[8:].abs().sum().item():.2e}")

loss = 441.741  (seg 399.437 + pose 42.305)
  grids   gradyanı sıfır olmayan: 3/3 | en küçük norm 7.2e-02
  inject  gradyanı sıfır olmayan: 3/6 | en küçük norm 0.0e+00
  ups     gradyanı sıfır olmayan: 12/12 | en küçük norm 1.3e+00
  skips   gradyanı sıfır olmayan: 12/12 | en küçük norm 2.4e+00
  head    gradyanı sıfır olmayan: 2/2 | en küçük norm 1.9e+02
Izgara gradyanı — batch'teki pair'ler: 3.03e+01 | batch dışındakiler: 0.00e+00


# **TRAINING**

In [55]:
import time
from tqdm import tqdm

EPOCHS, BATCH, EVAL_EVERY = 300, 16, 25
LR = {"net":1e-3, "grids":1e-2}
WARMUP = 100

net_params = [p for n, p in model.named_parameters() if not n.startswith("grids")]
opt = torch.optim.Adam([{"params": net_params, "lr": LR["net"]},
                        {"params": list(model.grids.parameters()), "lr": LR["grids"]}])
total = EPOCHS * math.ceil(len(PILOT) / BATCH)
lr_at = lambda s: min(1.0, (s + 1) / WARMUP) * 0.5 * (1 + math.cos(math.pi * min(s / total, 1.0)))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_at)


t0 = time.time()
for ep in tqdm(range(1, EPOCHS +1)):
  model.train()
  perm = PILOT[torch.randperm(len(PILOT), device=device)]
  ep_loss = 0.0

  for s in range(0, len(perm), BATCH):
    ids = perm[s:s + BATCH]
    loss, _, _ = task_loss(model(ids), ids)
    opt.zero_grad(set_to_none = True)
    loss.backward()
    nn.utils.clip_grad_norm(model.parameters(), 1.0)
    opt.step()
    sched.step()
    ep_loss += loss.item()
  if ep == 1 or ep % EVAL_EVERY ==0:
      r = score(lambda i: model(i), PILOT)
      print(f"E{ep:03d} | loss {ep_loss / math.ceil(len(PILOT) / BATCH):8.3f} | dist {r['dist']:.4f} "
              f"(seg {r['seg']:.4f}, pose {r['pose']:.4f}) | {(time.time() - t0) / 60:.1f} dk", flush=True)


  0%|          | 0/300 [00:00<?, ?it/s]/tmp/ipykernel_4019/1490302256.py:27: FutureWarning: `torch.nn.utils.clip_grad_norm` is now deprecated in favor of `torch.nn.utils.clip_grad_norm_`.
  nn.utils.clip_grad_norm(model.parameters(), 1.0)


E001 | loss  414.873 | dist 93.5174 (seg 50.6017, pose 42.9156) | 0.0 dk


  0%|          | 1/300 [00:03<18:30,  3.72s/it]


KeyboardInterrupt: 

LR deneyi

In [65]:
def run_pilot(levels, widths, epochs = 300, tag = "", batch = 20, lr_net=1e-3, lr_grid=1e-2, eval_every=100):
  torch.manual_seed(0)                                       # her deney aynı başlangıçla → adil kıyas
  m = GridDecoder(NUM_PAIRS, levels, widths).to(device)
  n_net = sum(p.numel() for n, p in m.named_parameters() if not n.startswith("grids"))
  n_grid = sum(g[0].numel() for g in m.grids.values())
  print(f"[{tag}] ağ {n_net:,} parametre | pair başına {n_grid:,} ızgara değeri")
  net_p = [p for n, p in m.named_parameters() if not n.startswith("grids")]
  opt = torch.optim.Adam([{"params": net_p, "lr": lr_net},
                          {"params": list(m.grids.parameters()), "lr": lr_grid}])
  total = epochs * math.ceil(len(PILOT) / batch)
  sched = torch.optim.lr_scheduler.LambdaLR(
      opt, lambda s: min(1.0, (s + 1) / 100) * 0.5 * (1 + math.cos(math.pi * min(s / total, 1.0))))
  t0 = time.time()

  for ep in tqdm(range(1, epochs + 1)):
    m.train()
    perm = PILOT[torch.randperm(len(PILOT), device=device)]
    for s in range(0, len(perm), batch):
            ids = perm[s:s + batch]
            loss, _, _ = task_loss(m(ids), ids)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
            sched.step()
    if ep % eval_every == 0:
            r = score(lambda i: m(i), PILOT)
            print(f"[{tag}] E{ep:04d} | dist {r['dist']:.4f} (seg {r['seg']:.4f}, pose {r['pose']:.4f}) | "
                  f"{(time.time() - t0) / 60:.1f} dk", flush=True)
  torch.save(m.state_dict(), OUT_DIR / f"pilot_{tag}.pt")
  del m, opt
  torch.cuda.empty_cache()
  return r

8-4 bit olcum

In [66]:
@torch.no_grad()
def ptq_net(state, bits):
    """Izgaralar FP32 kalır; sadece sentez ağının conv ağırlıkları kanal başına simetrik `bits` bite yuvarlanır."""
    m = GridDecoder(NUM_PAIRS).to(device)
    m.load_state_dict(state)
    if bits < 32:
        for mod in m.modules():
            if isinstance(mod, nn.Conv2d):
                w = mod.weight
                qmax = 2 ** (bits - 1) - 1
                s = w.abs().amax(dim=(1, 2, 3), keepdim=True).clamp_min(1e-12) / qmax
                w.copy_(torch.round(w / s).clamp(-qmax, qmax) * s)
    r = score(lambda i: m(i), PILOT)
    del m
    return r


state = torch.load(OUT_DIR / "pilot_uzun.pt", map_location=device)
base = ptq_net(state, 32)
print(f"FP32     | dist {base['dist']:.4f}")
for b in (8, 6, 5, 4):
    r = ptq_net(state, b)
    print(f"{b} bit ağ | dist {r['dist']:.4f} | kayıp {r['dist'] - base['dist']:+.4f} | ağ ≈ {45.4 * b / 8:.0f} KB")

FP32     | dist 0.1651
8 bit ağ | dist 0.1682 | kayıp +0.0031 | ağ ≈ 45 KB
6 bit ağ | dist 0.2187 | kayıp +0.0536 | ağ ≈ 34 KB
5 bit ağ | dist 0.2996 | kayıp +0.1345 | ağ ≈ 28 KB
4 bit ağ | dist 0.7974 | kayıp +0.6323 | ağ ≈ 23 KB


izgara olcum

In [67]:
rate_of_kb = lambda kb: 25 * kb * 1e3 / 37545489

@torch.no_grad()
def quant_net_(m, bits=8):
    """Sentez ağının conv ağırlıklarını yerinde, kanal başına simetrik `bits` bite yuvarlar."""
    qmax = 2 ** (bits - 1) - 1
    for mod in m.modules():
        if isinstance(mod, nn.Conv2d):
            s = mod.weight.abs().amax(dim=(1, 2, 3), keepdim=True).clamp_min(1e-12) / qmax
            mod.weight.copy_(torch.round(mod.weight / s).clamp(-qmax, qmax) * s)


def entropy_kb(q):
    """Tamsayı dizisinin ampirik entropisi (değerler bağımsız varsayılır) → KB."""
    _, counts = torch.unique(q, return_counts=True)
    p = counts.double() / q.numel()
    return float(-(counts * p.log2()).sum()) / 8 / 1e3


state = torch.load(OUT_DIR / "pilot_uzun.pt", map_location=device)
for k in ("64", "32", "16"):
    g = state[f"grids.{k}"][PILOT]
    print(f"Izgara 1/{k}: std {g.std():.3f} | ortalama |değer| {g.abs().mean():.3f} | max {g.abs().max():.2f}")

for frac in (0.25, 0.5, 1.0):                                   # adım = frac × o seviyenin std'si
    m = GridDecoder(NUM_PAIRS).to(device)
    m.load_state_dict(state)
    quant_net_(m, 8)
    kb, zero = 0.0, []
    with torch.no_grad():
        for k, g in m.grids.items():
            step = frac * g[PILOT].std()
            q = torch.round(g[PILOT] / step)
            g[PILOT] = q * step
            kb += entropy_kb(q)
            zero.append(f"%{100 * (q == 0).float().mean():.0f}")
    r = score(lambda i: m(i), PILOT)
    print(f"adım {frac:.2f}·std | dist {r['dist']:.4f} | ızgara {kb:.1f} KB (60 pair) → ~{10 * kb:.0f} KB (600 pair) "
          f"| sıfır oranı {'/'.join(zero)} | toplam ≈ {r['dist'] + rate_of_kb(10 * kb + 45):.3f}")
    del m

Izgara 1/64: std 0.489 | ortalama |değer| 0.386 | max 1.89
Izgara 1/32: std 0.450 | ortalama |değer| 0.351 | max 2.22
Izgara 1/16: std 0.432 | ortalama |değer| 0.329 | max 2.52
adım 0.25·std | dist 0.1995 | ızgara 60.9 KB (60 pair) → ~609 KB (600 pair) | sıfır oranı %12/%10/%12 | toplam ≈ 0.635
adım 0.50·std | dist 0.2768 | ızgara 46.0 KB (60 pair) → ~460 KB (600 pair) | sıfır oranı %23/%21/%23 | toplam ≈ 0.613
adım 1.00·std | dist 0.9433 | ızgara 31.5 KB (60 pair) → ~315 KB (600 pair) | sıfır oranı %44/%41/%43 | toplam ≈ 1.183


# **BIT CEZALI TRAINING**

In [68]:
STEP0 = 0.25


def laplace_bits(y, b):
  a = y.abs()
  a_in, a_out = a.clamp(max = 0.5), a.clamp(min = 0.50)

  p_in = 1 - 0.5 * (torch.exp(-(0.5 - a_in) / b) - torch.exp(-(0.5 + a_in) / b))
  p_out = 0.5 * (torch.exp(-(a_out - 0.5) / b) - torch.exp(-(a_out + 0.5)/b))

  return -torch.log2(torch.where(a<0.5, p_in, p_out).clamp_min(1e-9))
def grids_for(m, ids, train):
    """Eğitimde: ileri geçiş yuvarlanmış değeri görür (STE); bit tahmini için gürültülü kopya.
       Değerlendirmede: düz yuvarlama."""
    fwd, noisy = {}, {}
    for k, g in m.grids.items():
        x = g[ids]
        fwd[k] = x + (torch.round(x) - x).detach() if train else torch.round(x)
        noisy[k] = x + torch.rand_like(x) - 0.5
    return fwd, noisy

In [69]:
m = GridDecoder(NUM_PAIRS).to(device)
m.load_state_dict(torch.load(OUT_DIR / "pilot_uzun.pt", map_location=device))
with torch.no_grad():                                    # aynı fonksiyon, yeni birim: ızgara ÷ s, inject × s
    for k in m.grids:
        m.grids[k].div_(STEP0)
        m.inject[k].weight.mul_(STEP0)
log_b = nn.ParameterDict({k: nn.Parameter(torch.zeros(g.shape[1], device=device)) for k, g in m.grids.items()})
r = score(lambda i: m(i, grids_for(m, i, train=False)[0]), PILOT)
print("Başlangıç (yuvarlanmış ızgara, eğitimsiz):", r)

Başlangıç (yuvarlanmış ızgara, eğitimsiz): {'dist': 0.3201360357337023, 'seg': 0.2554575602213542, 'pose': 0.06467847551234814}


In [ ]:
EPOCHS, BATCH, EVAL_EVERY = 600, 8, 50
LAMBDA = 3.0                          # bit cezası çarpanı (eğitim loss'u ölçülen dist'ten ~3 kat büyük)
RAMP_EP = 100                         # ceza 100 epoch'ta 0'dan tam güce çıkar
BIT2SCORE = 25 / 8 / 37545489         # 1 bitin skordaki karşılığı
NET_KB = 45


net_p = [p for n, p in m.named_parameters() if not n.startswith("grids")]
opt = torch.optim.Adam([{"params": net_p, "lr": 3e-4},
                        {"params": list(m.grids.parameters()), "lr": 3e-2},
                        {"params": list(log_b.parameters()), "lr": 1e-2}])
total = EPOCHS * math.ceil(len(PILOT) / BATCH)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: min(1.0, (s + 1) / 50) * 0.5 * (1 + math.cos(math.pi * min(s / total, 1.0))))

t0 = time.time()
for ep in tqdm(range(1, EPOCHS + 1)):
  lam = LAMBDA * min(1.0, ep / RAMP_EP)

  m.train()
  perm = PILOT[torch.randperm(len(PILOT), device=device)]
  for s in range(0, len(perm), BATCH):
    ids = perm[s:s + BATCH]
    fwd, noisy = grids_for(m, ids, train=True)
    task, _, _ = task_loss(m(ids, fwd), ids)
    bits = sum(laplace_bits(noisy[k], log_b[k].exp().clamp_min(0.05).view(1, -1, 1, 1)).sum() for k in noisy)
    rate = bits / len(ids) * NUM_PAIRS * BIT2SCORE
    opt.zero_grad(set_to_none=True)
    (task + lam * rate).backward()
    torch.nn.utils.clip_grad_norm_(list(m.parameters()) + list(log_b.parameters()), 1.0)
    opt.step()
    sched.step()

  if ep % EVAL_EVERY == 0:
    r = score(lambda i: m(i, grids_for(m, i, train=False)[0]), PILOT)
    with torch.no_grad():
      q = {k: torch.round(g[PILOT]) for k, g in m.grids.items()}
      kb = sum(entropy_kb(v) for v in q.values()) * NUM_PAIRS / len(PILOT)
      zero = "/".join(f"%{100 * (v == 0).float().mean():.0f}" for v in q.values())
    print(f"E{ep:03d} λ={lam:.1f} | dist {r['dist']:.4f} (seg {r['seg']:.4f}, pose {r['pose']:.4f}) | "
              f"ızgara ~{kb:.0f} KB (600 pair) | sıfır {zero} | toplam ≈ {r['dist'] + rate_of_kb(kb + NET_KB):.3f} | "
              f"{(time.time() - t0) / 60:.1f} dk", flush=True)


torch.save({"model": m.state_dict(), "log_b": log_b.state_dict(), "step0": STEP0}, OUT_DIR / "pilot_rate_l3.pt")



  8%|▊         | 49/600 [01:18<14:35,  1.59s/it]

E050 λ=1.5 | dist 0.3263 (seg 0.2292, pose 0.0970) | ızgara ~434 KB (600 pair) | sıfır %30/%25/%23 | toplam ≈ 0.645 | 1.3 dk


 16%|█▋        | 99/600 [02:38<13:18,  1.59s/it]

E100 λ=3.0 | dist 0.3423 (seg 0.2259, pose 0.1164) | ızgara ~415 KB (600 pair) | sıfır %40/%33/%24 | toplam ≈ 0.649 | 2.7 dk


 25%|██▍       | 149/600 [03:59<11:58,  1.59s/it]

E150 λ=3.0 | dist 0.2911 (seg 0.2077, pose 0.0834) | ızgara ~390 KB (600 pair) | sıfır %48/%41/%28 | toplam ≈ 0.581 | 4.0 dk


... (15 outputs left out)

E550 λ=3.0 | dist 0.1716 (seg 0.1408, pose 0.0309) | ızgara ~326 KB (600 pair) | sıfır %63/%57/%34 | toplam ≈ 0.419 | 14.7 dk


100%|█████████▉| 599/600 [16:01<00:01,  1.59s/it]

E600 λ=3.0 | dist 0.1522 (seg 0.1285, pose 0.0237) | ızgara ~325 KB (600 pair) | sıfır %63/%57/%34 | toplam ≈ 0.399 | 16.1 dk


100%|██████████| 600/600 [16:04<00:00,  1.61s/it]


In [70]:
import time
from tqdm import tqdm
EPOCHS, BATCH, EVAL_EVERY = 600, 20, 50
LAMBDA = 3.0                          # bit cezası çarpanı (eğitim loss'u ölçülen dist'ten ~3 kat büyük)
RAMP_EP = 100                         # ceza 100 epoch'ta 0'dan tam güce çıkar
BIT2SCORE = 25 / 8 / 37545489         # 1 bitin skordaki karşılığı
NET_KB = 45


net_p = [p for n, p in m.named_parameters() if not n.startswith("grids")]
opt = torch.optim.Adam([{"params": net_p, "lr": 3e-4},
                        {"params": list(m.grids.parameters()), "lr": 3e-2},
                        {"params": list(log_b.parameters()), "lr": 1e-2}])
total = EPOCHS * math.ceil(len(PILOT) / BATCH)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: min(1.0, (s + 1) / 50) * 0.5 * (1 + math.cos(math.pi * min(s / total, 1.0))))

t0 = time.time()

In [71]:
def run_rate(lam_max, epochs, init_path, tag, ramp_ep=50, eval_every=50, lr_carpan=1.0,
             lam_min=3.0, batch=8, warm=50):
    ck = torch.load(init_path, map_location=device)
    m = GridDecoder(NUM_PAIRS).to(device); m.load_state_dict(ck["model"])
    log_b = nn.ParameterDict({k: nn.Parameter(v.detach().clone().to(device)) for k, v in ck["log_b"].items()})
    net_p = [p for n, p in m.named_parameters() if not n.startswith("grids")]
    grid_p = list(m.grids.parameters())
    opt = torch.optim.Adam([{"params": net_p, "lr": 3e-4 * lr_carpan},
                            {"params": grid_p, "lr": 3e-2 * lr_carpan},
                            {"params": list(log_b.parameters()), "lr": 1e-2 * lr_carpan}])
    adim = math.ceil(len(PILOT) / batch)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / (warm * adim))
                                              * 0.5 * (1 + math.cos(math.pi * min(1, s / (epochs * adim)))))
    for ep in tqdm(range(epochs)):
        lam = lam_min + (lam_max - lam_min) * min(1, (ep + 1) / ramp_ep)     # λ rampası
        perm = PILOT[torch.randperm(len(PILOT), device=device)]
        for i in range(0, len(perm), batch):
            ids = perm[i:i + batch]
            fwd, noisy = grids_for(m, ids, train=True)
            task, _, _ = task_loss(m(ids, grids=fwd), ids)
            bits = sum(laplace_bits(noisy[k], log_b[k].exp().clamp_min(0.05).view(1, -1, 1, 1)).sum()
                       for k in noisy)
            loss = task + lam * bits / len(ids) * NUM_PAIRS * BIT2SCORE
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(net_p + grid_p, 1.0); opt.step(); sched.step()
        if (ep + 1) % eval_every == 0 or ep + 1 == epochs:
            r = score(lambda ids: m(ids, grids=grids_for(m, ids, train=False)[0]), PILOT)
            q = {k: torch.round(g[PILOT].detach()) for k, g in m.grids.items()}
            kb = sum(entropy_kb(v) for v in q.values()) * NUM_PAIRS / len(PILOT)
            sifir = "/".join(f"%{100 * (v == 0).float().mean():.0f}" for v in q.values())
            print(f"[{tag}] E{ep + 1:03d} λ={lam:.0f} | dist {r['dist']:.4f} | ızgara ~{kb:.0f} KB "
                  f"| sıfır {sifir} | toplam ≈ {r['dist'] + rate_of_kb(kb + 45):.3f}")
    torch.save({"model": m.state_dict(), "log_b": log_b.state_dict(), "step0": ck.get("step0")},
               OUT_DIR / f"pilot_rate_{tag}.pt")
    return r, kb
def net_ft(init_path, tag, epochs=200, batch=8, lr=3e-4, warm=10):
    ck = torch.load(init_path, map_location=device)
    m = GridDecoder(NUM_PAIRS).to(device); m.load_state_dict(ck["model"])
    with torch.no_grad():
        for g in m.grids.values(): g.round_()
    m.grids.requires_grad_(False)
    kb = sum(entropy_kb(m.grids[k][PILOT]) for k in m.grids) * NUM_PAIRS / len(PILOT)
    params = [p for n, p in m.named_parameters() if not n.startswith("grids")]
    adim = math.ceil(len(PILOT) / batch)
    opt = torch.optim.Adam(params, lr=lr)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / (warm * adim))
                                              * 0.5 * (1 + math.cos(math.pi * min(1, s / (epochs * adim)))))
    for ep in tqdm(range(epochs)):
        perm = PILOT[torch.randperm(len(PILOT), device=device)]
        for i in range(0, len(perm), batch):
            ids = perm[i:i + batch]
            loss, _, _ = task_loss(m(ids), ids)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); sched.step()
        if (ep + 1) % 50 == 0:
            r = score(m, PILOT)
            print(f"[{tag}] E{ep + 1:03d} | dist {r['dist']:.4f} | ızgara {kb:.0f} KB "
                  f"| toplam {r['dist'] + rate_of_kb(kb + 45):.3f}")
    torch.save({"model": m.state_dict(), "log_b": ck["log_b"], "step0": ck.get("step0")},
               OUT_DIR / f"pilot_rate_{tag}.pt")
    return m, r, kb

In [72]:
sonuc = {}


In [74]:
PILOT = torch.arange(200, device=device)
print(len(PILOT))          # 200 görmelisin
m, r, kb = net_ft(OUT_DIR / "pilot_rate_p200_l100.pt", "p200_netft600", epochs=600, batch = 16)
m, r, kb = net_ft(OUT_DIR / "pilot_rate_p200_netft600.pt", "p200_netft900", epochs=300, lr=1e-4, batch = 16)

200


  8%|▊         | 50/600 [04:42<58:13,  6.35s/it]

[p200_netft600] E050 | dist 0.3893 | ızgara 78 KB | toplam 0.471


 17%|█▋        | 100/600 [09:26<52:56,  6.35s/it]

[p200_netft600] E100 | dist 0.3560 | ızgara 78 KB | toplam 0.438


 25%|██▌       | 150/600 [14:10<47:37,  6.35s/it]

... (27 outputs left out)

 83%|████████▎ | 250/300 [23:40<05:17,  6.35s/it]

[p200_netft900] E250 | dist 0.2977 | ızgara 78 KB | toplam 0.380


100%|██████████| 300/300 [28:24<00:00,  5.68s/it]

[p200_netft900] E300 | dist 0.2972 | ızgara 78 KB | toplam 0.379


In [75]:
PILOT = torch.arange(60, device=device)
sd = torch.load(OUT_DIR / "pilot_uzun.pt", map_location=device)
for k in GRID_LEVELS:
    sd[f"grids.{k}"] /= STEP0
    sd[f"inject.{k}.weight"] *= STEP0
log_b = {str(k): torch.zeros(sd[f"grids.{k}"].shape[1], device=device) for k in GRID_LEVELS}
torch.save({"model": sd, "log_b": log_b, "step0": STEP0}, OUT_DIR / "pilot_rate_p60tek_init.pt")

run_rate(100, 600, OUT_DIR / "pilot_rate_p60tek_init.pt", "p60tek", ramp_ep=200, eval_every=100)
net_ft(OUT_DIR / "pilot_rate_p60tek.pt", "p60tek_ft600", epochs=600)
net_ft(OUT_DIR / "pilot_rate_p60tek_ft600.pt", "p60tek_ft900", epochs=300, lr=1e-4)

 17%|█▋        | 100/600 [02:39<15:03,  1.81s/it]

[p60tek] E100 λ=52 | dist 0.4139 | ızgara ~186 KB | sıfır %83/%79/%52 | toplam ≈ 0.568


 33%|███▎      | 200/600 [05:19<12:07,  1.82s/it]

[p60tek] E200 λ=100 | dist 0.4833 | ızgara ~77 KB | sıfır %95/%93/%66 | toplam ≈ 0.565


 50%|█████     | 300/600 [08:01<09:04,  1.82s/it]

[p60tek] E300 λ=100 | dist 0.4614 | ızgara ~58 KB | sıfır %96/%96/%70 | toplam ≈ 0.530


... (39 outputs left out)

[p60tek_ft900] E250 | dist 0.2154 | ızgara 49 KB | toplam 0.278


100%|██████████| 300/300 [07:54<00:00,  1.58s/it]

[p60tek_ft900] E300 | dist 0.2153 | ızgara 49 KB | toplam 0.278


(GridDecoder(
   (grids): ParameterDict(
       (16): Parameter containing: [torch.cuda.FloatTensor of size 600x2x24x32 (cuda:0)]
       (32): Parameter containing: [torch.cuda.FloatTensor of size 600x2x12x16 (cuda:0)]
       (64): Parameter containing: [torch.cuda.FloatTensor of size 600x2x6x8 (cuda:0)]
   )
   (inject): ModuleDict(
     (64): Conv2d(2, 16, kernel_size=(1, 1), stride=(1, 1))
     (32): Conv2d(2, 16, kernel_size=(1, 1), stride=(1, 1))
     (16): Conv2d(2, 16, kernel_size=(1, 1), stride=(1, 1))
   )
   (ups): ModuleList(
     (0-2): 3 x Conv2d(16, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
     (3): Conv2d(16, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
     (4): Conv2d(12, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
     (5): Conv2d(12, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
   )
   (skips): ModuleList(
     (0-2): 3 x Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
     (3): Conv2d(16, 12, kernel_size=(1, 1), stride=

In [76]:
for ad in ["pilot_uzun.pt", "pilot_p200.pt"]:
    sd = torch.load(OUT_DIR / ad, map_location=device)
    satir = []
    for k in GRID_LEVELS:
        g = sd[f"grids.{k}"][:60]                      # aynı ilk 60 pair
        q = torch.round(g / STEP0)
        satir.append(f"1/{k}: std {g.std():.3f} sıfır %{100 * (q == 0).float().mean():.0f} "
                     f"{entropy_kb(q) * 10:.0f} KB")
    print(f"{ad:16s} | " + " | ".join(satir))

pilot_uzun.pt    | 1/64: std 0.489 sıfır %23 22 KB | 1/32: std 0.450 sıfır %23 83 KB | 1/16: std 0.432 sıfır %27 326 KB
pilot_p200.pt    | 1/64: std 0.650 sıfır %16 25 KB | 1/32: std 0.629 sıfır %17 97 KB | 1/16: std 0.609 sıfır %17 384 KB
